**Coursebook: SQL Query**
- Part of Data Science in Python Specialization for PT Freeport Indonesia
- Course Length: 12 hours
- Last Updated: September 2025
___

- Developed by [Algoritma](https://algorit.ma)'s product division and instructors team

# Background

## Top-Down Approach 

The coursebook is part of the Data Science in Python Specialization for PT Freeport Indonesia offered by [Algoritma](https://algorit.ma). The coursebook is intended for a restricted audience only, i.e. the individuals and organizations having received this coursebook directly from the training organization. It may not be reproduced, distributed, translated or adapted in any form outside these individuals and organizations without permission. 

Algoritma is a data science education center based in Jakarta. We organize workshops and training programs to help working professionals and students gain mastery in various data science sub-fields: data visualization, machine learning, data modeling, statistical inference etc.

## Training Objectives

This coursebook is intended for participants who have completed the preceding courses.

The coursebook focuses on:

**Module 1: Relational Databases Concept**

- Introduction to Entity Relationship Diagram (ERD)

**Module 2: Working with SQL Query**

- Introduction to SELECT Statements
- SQL Join Table
- SQL Aggregation
- WHERE Statements
- Using LIKE Operator
- HAVING Statement

**Module 3: Performance Query**

- Sub Query?
  - Purpose & Concept of Sub Query
  - Functionality Sub Query within SQL Syntax (SELECT, JOIN, WHERE, etc)
- Temporary Table
  - Purpose & Concept of Temporary Table
  - How to Create a Temporary Table for Storing Data?
  - How to access Temporary Table
  - Comparison Between Sub Query & Temporary Table
- Window Function
  - Aggregate
  - Ranking
  - Value


# Working with SQL Databases

There are a great number of python modules that provide functionalities to work with databases of all variants and flavors. For a MySQL database, we may form a connection using `pymysql` or one of many other alternatives:

```python
import pymysql

conn = pymysql.connect(
    host=host,
    port=port,
    user=user,
    password=password,
    db=database)
```

We can then use `pd.read_sql_query()`, passing in the connection:
```
sales = pd.read_sql_query("SELECT * FROM sales", conn)
```

Under the hood, `pandas` uses SQLAlchemy so any database supported by that library will work. This isn't something you need to worry about at this stage of your learning journey, but for the sake for practice, let's also see how a connection URI for a SQLite database looks like:

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data_input/chinook.db")

albums = pd.read_sql_query("SELECT * FROM albums", conn)
albums.head()

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data_input/chinook.db")

albums = pd.read_sql_query("SELECT * FROM employees", conn)
albums.head()

In the above command, we asked for all columns of a table to be returned to us through the `SELECT *` command. Well, columns of which table? That would be `tables`. Together they form an SQL query:

`SELECT * FROM albums`

The database we're working with have a few tables populated with sample data. The database has the following schema:

![](assets/chinookschema.png)

#### Knowledge Check

We'll create a `DataFrame`: this time select all columns from the `artists` table. Recall that when we use `pd.read_sql_query()` command we pass in the SQL query as a string, and add a connection as the second parameter. Save the output as a `DataFrame`.

Your DataFrame should be constructed like this:

`__ = pd.read_sql_query("SELECT __ FROM __ ", conn)`

Question:
1. How many rows are there in your DataFrame?

In [ ]:
## Your code below
artists = pd.read_sql_query("SELECT * FROM artists", conn)
artists
## -- Solution code

The `pd.read_sql_query` is most commonly used with that two parameters above, but on its [official documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_sql_query.html) is a list of other parameters we can use as well. 

In the following cell, we use a similar SQL query with an additional `LIMIT` statement to limit the output to the first 5 records (rows). However, notice that we also set `index_col` so the specified column is recognized as the index:

In [ ]:
pd.read_sql_query("SELECT * FROM artists LIMIT 5", 
                  conn, 
                  index_col='ArtistId')

## SQL Joins

`JOIN` statements are used to combine records from two tables. We can have as many `JOIN` operations as we want in a SQL query.

Below is a diagram of the different types of SQL `JOIN` operations:

![](assets/sqljoins.png)

Credit: Data & Object Factory, LLC

In most business scenarios though, a `LEFT JOIN` is almost always the type of `JOIN` you want - it is very direct (and therefore easy to reason about). Left join return all records in the left table regardless if any of those records have a match in the right table.

The `INNER JOIN` is also very intuitive and easily understood. This query return all of the records in the left table that has a matching record in the right table.

> As a personal side note, I've worked at companies where `RIGHT JOIN` is outright forbidden in favor of `LEFT JOIN`: directness and ease-of-understanding aside, all right joins can be replaced by the opposite left join. 
> 
> The `OUTER JOIN` (also referred to as `FULL JOIN`) is also quite uncommon in practice. Performance reason aside, an outer join return all of the records from both tables regardless if there is a match or not, resulting in a DataFrame that has potentially a lot of `NULL` values.

Consider the database schema illustraation again and pay attention to two tables and their respective columns:

1. `albums`: 
    - `AlbumId`, `Title`, `ArtistId`

2. `artists`: `
    - `ArtistId`, `Name`    

We want a `pandas` DataFrame containing the `AlbumId`, `Title` and `Name`. Notice that `Name` is from the `artists` table while the other columns are from the `albums` table. What is a reasonable strategy?

The most straightforward solution is the `LEFT JOIN`, let's see an example:

In [ ]:
albums = pd.read_sql_query("SELECT AlbumId, Title, a.Name \
                           FROM albums \
                           LEFT JOIN artists as a \
                           ON a.ArtistId = albums.ArtistId", conn)
albums.head()

Notice that in the code above, we place a backslash (`\`) character so we have line continuation and the newline will be ignored. This allows SQL to treat the entire query string as if they were essentially one line.

In [ ]:
pd.read_sql_query("SELECT * FROM tracks", conn).head()

In [ ]:
pd.read_sql_query("SELECT tracks.*,artists.Name as artist_name FROM tracks LEFT JOIN albums ON tracks.AlbumId = albums.AlbumId LEFT JOIN artists ON albums.ArtistId = artists.ArtistId", conn)

#### Knowledge Check

Consider the database schema illustraation again and pay attention to two tables and their respective columns:

1. `albums`: `AlbumId`, `Title`, `ArtistId`

2. `tracks`: `TrackId`, `Name`, `AlbumId`, `GenreId`, ... `UnitPrice` 
    
3. `genres`: `GenreId`, `Name`

Create a `DataFrame` containing all columns from the `tracks` table; Additionally, it should also contain:
    - The `Title` column from the `albums` table
    - The `Name` column from the `artists` table
    - The `Name` column from the `genres` table 

> **Hint 1**: In your `SELECT` statement, you can use `SELECT tracks.* FROM TRACKS` to select all columns from the `TRACKS` table
> 
> **Hint 2**: When we write `SELECT tracks.Name as tracksName`, we are renaming the output column from `Name` to `tracksName` using a technique called column aliasing. You may optionally consider doing this for columns that share the same name across different tables 

Set the `TrackId`column to be the index. The resulting `DataFrame` should has 11 columns.

Give your `DataFrame` a name: name it `tracks`. Perform EDA on `tracks` to answer the following question:

1. Use `tail()` to inspect the last 5 rows of data. Which genre is present in the last 5 rows of our `tracks` DataFrame (Check all that apply)?
    - [ ] Latin
    - [ ] Classical
    - [ ] Soundtrack
    - [ ] Pop

2. Apply `pd.crosstab(..., columns='count')`, `.value_counts()`, or any other techniques you've learned to compute the frequency table of Genres in your DataFrame. Which is among the top 3 most represented genres in the `tracks` DataFrame?
    - [ ] Latin
    - [ ] Classical
    - [ ] Soundtrack
    - [ ] Pop

3. Use `groupby()` on Artist Name and compute the `mean()` on the `UnitPrice` of each tracks. You will realize that most artists price their tracks at 0.99 (`mean`) but there are several artists where the `mean()` is 1.99. Which of the Artist has a mean of 0.99 `UnitPrice`:
    - [ ] The Office
    - [ ] Aquaman
    - [ ] Pearl Jam
    - [ ] Lost

In [ ]:
## Your code below
tracks = pd.read_sql_query("SELECT * FROM tracks JOIN genres ON tracks.GenreId = genres.GenreId", conn)
tracks.tail()
## -- Solution code

In [ ]:
pd.crosstab(tracks['Name'],columns='count').sort_values(by='count',ascending=False).head(3)

In [ ]:
artist_all = pd.read_sql_query("SELECT tracks.*,artists.Name as artist_name FROM tracks LEFT JOIN albums ON tracks.AlbumId = albums.AlbumId LEFT JOIN artists ON albums.ArtistId = artists.ArtistId", conn)

In [ ]:
mean_artist = pd.DataFrame(artist_all.groupby('artist_name')['UnitPrice'].mean())
mean_artist[(mean_artist['UnitPrice'] == 0.99)&(mean_artist.index.isin(['The Office','Aquaman','Pearl Jam','Lost']))]

## SQL Aggregation
Since you have learned various aggregation tools in `pandas` such as `.crosstab()`, `.pivot_table()`, and `.groupby()`. It is also common practice to perform aggregation function using SQL. Consider the following query:

In [ ]:
top_cust = pd.read_sql_query("SELECT CustomerId, SUM(Total)  as TotalValue, \
                              COUNT(InvoiceId) as Purchases \
                              FROM INVOICES \
                              GROUP BY CustomerId \
                              ORDER BY TotalValue DESC \
                              LIMIT 5", con=conn, index_col='CustomerId')
top_cust

Notice how the query fetched the top 5 customers from all time from the `invoices` table grouped by unique `CustomerId`. We performed two aggregation functions: `SUM()` and `COUNT()`, each aggregating different column from the `invoices` table. At the end, the `ORDER BY` statement is added to order the table based on the `TotalValue` column. Do note that the aggregated columns needs to be a numeric as the available aggregated functions are: `SUM`, `AVG`, `COUNT`, `MIN`, and `MAX`.

#### Knowledge Check

Edit the following code to find out the most popular `genres` from all invoice sales. Use different column to acquire the following information: Summation of `Total` sales, and number of tracks bought from the `Quantity` columns.

hint: The `Total` can be obtained from `UnitPrice` and `Quantity` column in `invoice_items`

```python
top_genre = pd.read_sql_query("SELECT genres.GenreId, genres.Name, \ 
                               __(invoice_items.quantity * invoice_items.unitprice) AS Total, \
                               SUM(invoice_items.__) AS ___ \
                               FROM tracks \
                               LEFT JOIN genres ON _____ \
                               LEFT JOIN invoice_items ON ______ \
                               GROUP BY _____ \
                               "
                               conn,
                               index_col='GenreId'
)
```

Question:
1. What are the top 5 genres that generated the most profit?

In [ ]:
## Your code below
query = """
                              SELECT genres.GenreId, genres.Name,
                               SUM(invoice_items.quantity * invoice_items.unitprice) AS Total,
                               SUM(invoice_items.quantity) AS total_track
                               FROM tracks
                               LEFT JOIN genres ON tracks.GenreId = genres.GenreId
                               LEFT JOIN invoice_items ON tracks.TrackId = invoice_items.TrackId
                               GROUP BY genres.GenreId, genres.Name ORDER BY Total DESC LIMIT 5
"""
top_genre = pd.read_sql_query(query,
                               conn,
                               index_col='GenreId'
)
top_genre


## WHERE statements

We've seen how to use do some of the most common SQL operations this far. In particular, we have:

- Learned how to write `SELECT` statements  
- Use `index_col` in the `pd.read_sql_query()` method  
- SQL Join operations
- Use SQL Aliases

In the following example, we'll look at one more technique in the SQL arsenal: the `WHERE` clause

A `WHERE` clause is followed by a **condition**. If we want to query for all invoices where country of the billing address is Germany, we can add a `WHERE` clause to our sql query string:

In [ ]:
germany = pd.read_sql_query("SELECT * FROM invoices WHERE BillingCountry = 'Germany'", conn)
germany.head()

`WHERE` conditions can be combined with `IS`, `AND`, `OR` and `NOT`. Supposed we want to create a DataFrame containing all invoices where the billing country is **not** Germany, we can do the following:

In [ ]:
not_germany = pd.read_sql_query("SELECT * FROM invoices WHERE BillingCountry IS NOT 'Germany'", conn)
not_germany.head()

Do note that the `IS` operator is the same as its mathematical notation counterpart: `=`. Similar with most programming language, it also supports other mathematical operator such as `>`, `>=`, `<`, and `<=`.

Now let's try another approach by using `IN` operator that enables us to specify multiple values for comparison. For example we'd like to retrieve all invoices from `Canada` and `USA`:

In [ ]:
america = pd.read_sql_query("SELECT * FROM invoices WHERE BillingCountry IN ('USA', 'Canada')", conn)
america.head()

#### Knowledge Check

Edit the following code to include a `WHERE` clause. We want the returned DataFrame to contain only the `Pop` genre and only when the `UnitPrice` of the track is 0.99:

```python
popmusic = pd.read_sql_query("SELECT tracks.*, genres.Name as GenreName \
                            FROM tracks \
                            LEFT JOIN genres ON _____ \
                            WHERE genres.Name = ____ AND _____, 
                           conn,
                           index_col='TrackId'

)
```

Question:
1. How many rows are there in `popmusic`?

In [ ]:
## Your code below


## -- Solution code

### Operating Dates

Continuing from the last `WHERE` statements, we can retrieve all invoices billed to the country `Germany`. However, it is also common to perform a conditional query statement to retrieve a specific data range. Let's take a look at our `germany`data frame for example:

In [ ]:
germany.dtypes

Notice how our `InvoiceDate` is listed as `Object` types. The `pd.read_sql_query` behaves like `pd.read_csv` where, by default, it reads data as numeric and object. This doesn't necessarily means the database is stored using string format (commonly known as `VARCHAR` in SQL databases). Take a look at the following table schema:

In [ ]:
invoices_table = pd.read_sql_query("SELECT sql FROM sqlite_master \
                                    WHERE name = 'invoices'", conn)
print(invoices_table.loc[0,:].values[0])

Note that according to different DBMS you are using, there are different ways to retrieve a table's schema. The query above is used to retrieve table schema from SQLite database. The `DATETIME` type is stored with the following format: `YYYY-MM-DD HH:MI:SS`.

It is often useful to understand the table schema of your database so you can perform the appropriate operation. Within our `invoices`' schema you can see some of useful information such as:
- `InvoiceId` is listed as a primary key
- `InvoiceDate` is stored as `DATETIME`
- `CustomerId` is registered as foreign key to `customers` table

If you are not provided with the database's schema, take some time to study each table schema. Now consider the following case: We are reviewing Germany market of the last year sales and would like to retrieve all invoices from the year `2012`.

In [ ]:
germany_2012 = pd.read_sql_query("SELECT * FROM invoices \
                                  WHERE InvoiceDate >= '2012-01-01' AND InvoiceDate <= '2012-12-31'",
                                 con=conn, parse_dates='InvoiceDate')
germany_2012['InvoiceDate'].describe()

Now there are other common approach using the `BETWEEN` operator, try and copy-paste the following code to a code cell:

```python
germany_2012 = pd.read_sql_query("SELECT * FROM invoices \
                                    WHERE InvoiceDate BETWEEN \
                                    '___' AND '___'", 
                                  con=conn, parse_dates='InvoiceDate')
germany_2012['InvoiceDate'].describe()
```

Try completing the code above and see if the query fetch the same result as the previous one:

In [ ]:
# Your code below


### Using `LIKE` Operator

Using a `WHERE` statement in a query can be beneficial to pull relevant data for our analysis. A common operator for a WHERE statement is `LIKE`.
Consider the following SQL Query:

In [ ]:
germany = pd.read_sql_query("SELECT * FROM invoices \
                             WHERE BillingCountry = 'Germany' \
                             AND BillingPostalCode LIKE '107%'", conn)
germany.head()

A `LIKE` operator is a can be used to match a certain part of the data and can be really useful if we need to perform a partial matching to a specific value rather than using an equal-to operator. The `107%` you see in the query means to extract value in `BillingPostalCode` that starts with the number 107. This is really helpful when you are aiming to extract data specifically on specific region. In Germany, you would know that Wilmersdorf and Tempelhof in Berlin has postal code starting with 107.

**Discussion:**

If you were to put `%` on the end, you are matching everything that starts with 107, if you put it on the start `%107`, means you are matching everything that ends in 107. What do you think will came up if you use `%`  before and after the pattern match?

#### Dive Deeper:

Let's take a look at another case, consider the following data frame:

In [ ]:
customerinv = pd.read_sql_query("SELECT firstname, lastname, email, company, \
                                 invoiceid, invoicedate, billingcountry, total \
                                 FROM invoices \
                                 left join customers \
                                 on invoices.customerId = customers.customerId", conn)
customerinv.head()

Within the first 6 data, you could see the `Company` column may not be reliable since most of them are filled with None. But if you pay close attention to the `Email` column, you could see some people have an email domain at `apple`, that could be an indicator of their company.

1. How would your conditional `WHERE` statement would be like if you want to count the number of customers that are working at Apple Inc.?

    Try to complete the following codes:

```py
applecust = pd.read_sql_query("SELECT firstname, lastname, email, company, \
                               invoiceid, invoicedate, billingcountry, total \
                               FROM invoices \
                               left join customers \
                               on invoices.___ = customers.___ \
                               WHERE email like '___'", conn)

)
```

In [ ]:
## Your code below

query = """
    SELECT FirstName, LastName, Email, Company, InvoiceId, InvoiceDate, BillingCountry, Total
    FROM invoices
    LEFT JOIN customers
    ON invoices.CustomerId = customers.CustomerId
    WHERE Email LIKE '%@apple%'
"""

output = pd.read_sql_query(query, conn)

output
## -- Solution code

Based on the data queried, how many of the customers is working at Apple Inc.?

- [ ] 412  
- [ ] 49   
- [ ] 7  
- [ ] 14  

## HAVING Statement

We have learned about how to perform aggregation in SQL and the usage of the `WHERE` statement. However, it's important to note that you can't use the `WHERE` clause to filter rows based on conditions on an aggregated column.

In the following example, we'll explore another technique in SQL: the `HAVING` clause.

The key difference between `HAVING` and `WHERE` is that the `HAVING` clause is used after the aggregation function. If we want to retrieve customers with a total invoice amount from all time greater than 47, let's consider the query below:

In [ ]:
top_cust_47 = pd.read_sql_query("SELECT CustomerId, SUM(Total)  as TotalValue \
                              FROM INVOICES \
                              GROUP BY CustomerId \
                              HAVING TotalValue > 47 \
                              ORDER BY TotalValue DESC", con=conn, index_col='CustomerId')
top_cust_47

Notice how the query retrieves customers with a total invoice amount greater than 47 from all time. We performed an aggregation function to find the TotalValue for each CustomerId. We used the `HAVING` clause to filter the rows based on certain conditions in the aggregated column.

#### Knowledge Check

Edit the following code to include a `HAVING` clause. We want the dataframe to only contain composers who have sold more than 15 tracks.

```py
top_5_composer = pd.read_sql_query("SELECT tracks.____,  SUM(invoice_items.___) AS TrackSold\
                                    FROM tracks \
                                    JOIN ___ ON ____ =  ____ \
                                    GROUP BY tracks.______\
                                    HAVING ____ \
                                    ORDER BY ____ DESC ", 
                                    conn)
```

Question : 

1. How many Composer having TrackSold more than 15 ? 

In [ ]:
## Your code below

## -- Solution code

# SQL Execution Order

Up to this point, we have discussed various SQL queries, starting from `SELECT`, the simplest query for selecting which data to present, to `HAVING`, for subsetting grouped data. Although `SELECT`, for example, is frequently written at the beginning of the query, it is not executed first. Therefore, in this chapter, we will mainly discuss the SQL execution order to gain a deeper understanding of how a set of queries work together.

First, it is necessary to define from which source the data will be queried. Therefore, `FROM` and `JOIN` statements are executed first. This includes the subqueries associated with these statements. Next, if any condition is defined, then the `WHERE` statement will be executed afterward. The rows that satisfy this condition will be included, and those that do not will be discarded. Following this, when we need to group the data, the `GROUP BY` statement is executed. After grouping, if we want to filter the grouped data, the `HAVING` statement is used.

Finally, the `SELECT` statement is executed to determine which parts of the queried data we want to display. The subsequent statements are `ORDER BY` and `LIMIT`, which order and limit the data display, respectively.

In summary, the SQL execution order is:

`FROM` - `JOIN` - `WHERE` - `GROUP BY` - `HAVING` - `SELECT` - `ORDER BY` - `LIMIT`.

# Performance Query

When working with databases, query performance is a critical factor that determines how quickly and efficiently data can be retrieved or manipulated. Poorly optimized queries can slow down processes, overload the server, and negatively impact the overall performance of an application. Therefore, understanding how to write efficient queries, leverage indexes, and avoid costly operations is an essential skill.

## Sub-Query

In some cases, you'd like to fill in some value for the query from the database itself as one of the condition. For example, recall how we retrieved all customers that has the most total invoice in the previous exercise. Say from the information, we'd like to retrieve all the top customers invoice. To do that we will construct a `WHERE` statement using `IN` operator utilizing a subquery to retrieve all top customers:

In [ ]:
customerinv = pd.read_sql_query("""
    SELECT invoices.* 
    FROM invoices
    WHERE invoices.CustomerId IN (
        SELECT c.CustomerId 
        FROM Customers as c
        LEFT JOIN invoices as i on i.CustomerId = c.CustomerId
        GROUP BY c.CustomerId
        ORDER BY SUM(Total) 
        DESC LIMIT 10
    )
""", conn)
customerinv

Notice the subquery is defined as follow:

```sql
SELECT c.CustomerId FROM Customers as c 
LEFT JOIN invoices as i on i.CustomerId = c.CustomerId 
GROUP BY c.CustomerId 
ORDER BY SUM(Total) DESC LIMIT 10
```

If used on its own, the query will retrieve the top 10 customer's IDs based on the total summation of their purchases. This when then used as the condition using `IN` statement on the previous query. The `IN`, however, can also be used using a hard-coded value like the following:

In [ ]:
customerinv = pd.read_sql_query("SELECT *  \
                                 FROM invoices \
                                 WHERE InvoiceId IN (46, 175, 198)", conn)
customerinv.head()

#### Knowledge Check

Imagine you're being instructed to analyze invoices that consist of a fair amount of tracks within one purchase. Previously, you have known that within one invoice the users would see a total of 1 to 10 tracks per purchase. Using subquery techniques you have learned, create an SQL query that fetched all invoice along with its total tracks quantity of bigger than 10. Complete the following query:

```sql
SELECT *, 
(SELECT ___ FROM invoice_items GROUP BY ___) as Quantity
FROM invoices
WHERE Quantity > 10
```

In [ ]:
## Your code below


## -- Solution code

## Temporary Tables

Temporary tables are a powerful tool for storing intermediate results during complex queries. Imagine a whiteboard where you can jot down notes and calculations during a problem-solving session. Similarly, temporary tables provide a space to store and manipulate data without altering the main database tables. They exist only for the duration of your current SQL/Python session, acting as a temporary workspace for data transformations and aggregations.

Temporary tables are particularly useful when dealing with multi-step operations, such as joining multiple tables and performing complex aggregations. They help break down complex queries into smaller, more manageable parts, improving readability and maintainability. By storing intermediate results, you can avoid redundant computations and optimize query performance. 

To create a temporary table, the general structure is:

```sql
CREATE TEMPORARY TABLE temp_table_name AS
SELECT column1, column2, ...
FROM table1
...
```

Now, let's create a temporary table `temp_costumers` to retrieve all the top customers invoice like we did earlier using sub queries using `conn.execute()`:

In [ ]:
conn.execute('''
    CREATE TEMPORARY TABLE temp_customers AS
             
    SELECT c.CustomerId 
    FROM Customers as c
             
    LEFT JOIN invoices as i 
    ON i.CustomerId = c.CustomerId
             
    GROUP BY c.CustomerId
             
    ORDER BY SUM(Total) DESC 
             
    LIMIT 10
''')

And we can access the temporary table using `FROM` and get the information that we needed using `pd.read_sql_query()`:

In [ ]:
pd.read_sql_query('''
                  SELECT *
                  FROM temp_customers

                  LEFT JOIN invoices
                  ON invoices.CustomerId = temp_customers.CustomerId

                  ''', conn)

Notice that while both subqueries and temporary tables work, temporary tables offer a more structured and step-by-step approach. With subqueries, all the logic must be nested inside a single SQL statement, which can quickly become hard to read and debug as the complexity grows. On the other hand, temporary tables let you break the problem into multiple SQL statements.

## Window Function

Window functions are one of the most powerful features in SQL that allow you to perform calculations across a set of rows while still keeping each individual row in the result. Unlike aggregate functions such as SUM or AVG that collapse rows into a single output, window functions return a value for every row, providing context while preserving detail.

Window functions are defined in pairs with `OVER()` statement, where you can specify: 
- `PARTITION BY` to divide the data into groups.
- `ORDER BY` to set the order of rows within each group.

Where the general structure is:

```sql
SELECT column_name1, 
       WINDOW_FUNCTION(column_name2) OVER (PARTITION BY column_name3 ORDER BY column_name4) AS new_column
FROM table_name
```
- `WINDOW_FUNCTION`: Window functions to be applied
- `column_name1`: Regular column(s) to be selected in the output 
- `column_name2`: Column on which the window function is applied
- `column_name3`: Column used for dividing rows into groups (`PARTITION BY`)
- `column_name4`: Column used to define order of rows within each partition (`ORDER BY`)

Throughout this section, we will explore different types of window functions

## Aggregate Window Function

**Aggregate Window Functions** extend the behavior of traditional aggregate functions by applying them over each rows without collapsing the result into a single record, like how `GROUP BY` works. This means you can calculate group-level statistics while still preserving each individual row in the output.

For Aggregate Window Functions, you typically use the same aggregate functions you already know in SQL, but combined with the `OVER()` clause. Here are the most common ones:

- `SUM(column)`   : cumulative totals, running sums
- `AVG(column)`   : moving averages, average per group
- `COUNT(column)` : count of rows in a window
- `MIN(column)`   : smallest value in the window
- `MAX(column)`   : largest value in the window

For example, you might want to compute the total of sales for each customers:

In [ ]:
pd.read_sql_query('''
                  SELECT CustomerId, InvoiceDate, Total,
                         SUM(Total) OVER(PARTITION BY CustomerId) AS TotalPerCustomer
                  FROM invoices
                  ''', conn).head(20)

The window function will sum a sequence of values as it progresses through a dataset. calculating the sum of `Total` for each `CustomerId`.

## Ranking Window Functions

**Ranking Window Functions** are used to assign an order or ranking to rows within a defined partition of data. Unlike aggregate functions that summarize values, ranking functions help identify the relative position of each row based on sorting rules.

These are some of the most common Window Functions to be used for rankings:

- `ROW_NUMBER()` : Assigns a unique sequential number to each row in the result set (no duplicates, no gaps).
- `RANK()` : Assigns a rank starting from 1, but leaves gaps in ranking if there are ties.
- `DENSE_RANK()` : Similar to `RANK()`, but does not leave gaps when ties occur.
- `NTILE(n)` : Divides the rows into n roughly equal groups (tiles or buckets), and assigns each row to one of those groups.

Consider this following example to rank each customers invoices:

In [ ]:
pd.read_sql_query('''
                  SELECT CustomerId, InvoiceDate, Total,
                         RANK() OVER(PARTITION BY CustomerId ORDER BY Total ASC) AS RankingPerCustomer
                  FROM invoices
                  ''', conn).head(20)

Notice that some duplicates exist in the `Total` and is ranked with the same value, while the value below it has gaps between the rankings.

## Value Window Functions

**Value Window Functions**, allow you to directly reference values from other rows within a defined window of data. Instead of aggregating or ranking, these functions "look around" the current row to retrieve information. These functions are particularly useful in time-series analysis, customer journey tracking, and row-to-row comparisons where context from neighboring rows matters.

The most common examples of Value Window Functions are:

- `LAG(column, offset)` : returns the value from a previous row within the same partition.
- `LEAD(column, offset)` : returns the value from a following row within the same partition.
- `FIRST_VALUE(column)` : returns the first value in the ordered window.
- `LAST_VALUE(column)` : returns the last value in the ordered window.

Suppose we want to compare each customer's transaction to their previous transactions:

In [ ]:
pd.read_sql_query('''
                  SELECT CustomerId, InvoiceDate, BillingCountry, Total,
                         LAG(Total, 1) OVER(PARTITION BY CustomerId ORDER BY InvoiceDate ASC) AS PreviousTotal
                  FROM invoices
                  ''', conn).head(20)

If the current row doesn't have a previous row (e.g., the very first row in each partition), then SQL has nothing to return, so it outputs NaN.

# Under and Over Fetching

Now amidst all the tools selection available for data analysis, you will now ponder which one is better suited for you. To review let's recall what we have learned about Python features:
- Reading flat files (csv / sas files)
- Data cleansing and wrangling
- Exploratory analysis tools
- Visual exploratory tools

Now where does SQL fits in? First you will need to understand client-server architecture. 

![](assets/clientserver.png)

A bit different with Python when all your data operation is done on your local computer. When you worked with SQL, most likely you have a relational database stored remotely from your machine, usually a centralized database accessible to some clients.

When you perform a query, you execute a command to download the data to your computer. This downloading process require resources, and you need to effectively utilize the tools in order to minimize the overall cost.

**Discussion:**

You are instructed to perform an analysis for all `Rock` genre sales on the last year (2012). Consider this questions:

- Is it necessary for you to download all tracks table to your computer?
- Will you filter the `Rock` genre tracks using SQL WHERE statements or conditional subsetting using Python?
- Since we need the information of multiple tables, which one is more convenient, querying a joined table or two separate tables from the database?

Try if you can construct your most optimum query below:

In [ ]:
## Your code below

